# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — connect to the warehouse

Same DuckDB-over-Parquet pattern as notebook 03: nothing downloads, nothing loads into RAM until a query says so. Working month for this contract: **`month=2026-03`** — mid-panel, safe to iterate on. `2026-02` is loaded too, only to build one prior-month feature. The final released month (`2026-06`) is never touched here — it stays a sealed test window.

In [19]:
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_mar':    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_feb':    f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')",
    'query_90d':   f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = '2026-03'       # contract + verification month (mid-panel)
PREV_MONTH = '2026-02'  # prior month, prevmonth features only


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit: one content item (`content_hash_id`), inside one client (`client_hash_id`), summarized over one calendar month.** For this contract that month is `month=2026-03` (2026-03-01 to 2026-03-31) — mid-panel, not the sealed final month (2026-06). Same lane as weeks 1-2, decline-risk classification, now built on the full warehouse instead of the 30k-row starter slice.

**Tables:** `fact_content_daily_performance` at the `month=2026-03` partition (the verification slice) and `month=2026-02` (prior-month feature support only, not part of the verified slice), `dim_content` (for `content_created_at`), `dim_clients` (per-client history coverage, sanity-checked in Section 4), `fact_content_query_90d` (a fixed 90-day query-mix snapshot, used cautiously — see Section 4).

In [20]:
print(f"Unit: one content item x client, summarized over month={MONTH}")
print(f"Verification slice: {MONTH} (mid-panel)  |  prior-month support: {PREV_MONTH}")
print("Sealed test month, never touched for label logic: 2026-06")


Unit: one content item x client, summarized over month=2026-03
Verification slice: 2026-03 (mid-panel)  |  prior-month support: 2026-02
Sealed test month, never touched for label logic: 2026-06


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label / proxy:** `is_down` — 1 if this content's March impressions fell more than 20% versus February (`imp_mar < 0.8 * imp_feb`), else 0. Same shape as the `trend_direction` proxy from weeks 1-2, rebuilt myself here since the warehouse doesn't ship a trend column — it comes from the raw daily facts, not a pre-computed bucket.

**Feature (knowable before 2026-03-01):**
- `imp_feb`, `pos_feb` — February's impressions / avg position, fully elapsed before March starts.
- `content_age_days` — `2026-03-01` minus `content_created_at`, fixed at creation.
- `visible_queries`, `rare_share` — query-mix shape from `fact_content_query_90d`, a content trait rather than a monthly outcome (window-alignment caveat in Section 4).

**Context (join/group only, never a model input):** `content_hash_id`, `client_hash_id`, `report_date`.

**Excluded, and why:** GA4-derived columns (sessions, engagement, scroll) for any row where `ga4_data_available` is not `TRUE`. Those rows are zero-filled before a client's `ga4_data_start`, so a zero there means "not tracked yet," not "no engagement" — feeding it in would teach the model tracking coverage, not content quality. Verified with a query in Section 3.

In [21]:
print("Label: is_down = imp_mar < 0.8 * imp_feb  (built from raw facts, not shipped)")
print("Features: imp_feb, pos_feb, content_age_days, visible_queries, rare_share")
print("Context only: content_hash_id, client_hash_id, report_date")
print("Excluded: GA4 columns where ga4_data_available is not TRUE")


Label: is_down = imp_mar < 0.8 * imp_feb  (built from raw facts, not shipped)
Features: imp_feb, pos_feb, content_age_days, visible_queries, rare_share
Context only: content_hash_id, client_hash_id, report_date
Excluded: GA4 columns where ga4_data_available is not TRUE


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



### a. Grain check

If "one row = one content item x client x day" is true, no `(report_date, client_hash_id, content_hash_id)` combination repeats. Zero rows back below means the grain holds.

In [22]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_mar']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate (date, client, content) rows in {MONTH}: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) rows in 2026-03: 0


,report_date,client_hash_id,content_hash_id,c


### b. Row count + date span

The size and date range of the March slice, checked directly rather than assumed.

In [23]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_mar']}
""").df()
slice_stats


,n_rows,n_content,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### c. Availability — `ga4_data_available IS TRUE`

Proves the Section 2 exclusion with real numbers: how many March rows actually carry usable GA4 data.[link text](https://)

In [24]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_mar']}
""").df()
survive_pct = availability['ga4_available_rows'][0] / availability['total_rows'][0] * 100
print(availability)
print(f"Rows with usable GA4 data: {survive_pct:.1f}% -- the rest are the Section 2 exclusion, not zero engagement.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0
Rows with usable GA4 data: 4.2% -- the rest are the Section 2 exclusion, not zero engagement.


### d. Five-feature frame

Decision moment: `2026-03-01`. Every feature below is fixed before that date.

- `imp_feb` — knowable at the decision moment because February is fully elapsed before March starts.
- `pos_feb` — knowable at the decision moment, same fully-elapsed prior month.
- `content_age_days` — knowable at the decision moment because `content_created_at` is fixed at creation, always in the past.
- `visible_queries` — knowable in the sense that it describes the content's query-mix shape rather than March's outcome — flagged as a window-alignment risk in Section 4, not a clean guarantee.
- `rare_share` — same table, same caveat: a content trait, not a March-specific number.

Minimum-volume filter: kept only content with `imp_feb >= 100`, so the label isn't computed on noise.

In [25]:
data.columns.tolist()

['content_hash_id',
 'client_hash_id',
 'imp_feb',
 'pos_feb',
 'imp_mar',
 'visible_queries',
 'rare_share',
 'content_created_date',
 'content_age_days',
 'is_down']

In [26]:
feb_agg = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS imp_feb,
           AVG(gsc_avg_position) AS pos_feb
    FROM {TABLES['fact_feb']}
    GROUP BY 1, 2
    HAVING imp_feb >= 100
""").df()

mar_agg = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS imp_mar
    FROM {TABLES['fact_mar']}
    GROUP BY 1, 2
""").df()

qmix = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share) AS rare_share
    FROM {TABLES['query_90d']}
    GROUP BY 1
""").df()

age = con.sql(f"""
    SELECT content_hash_id, content_created_date
    FROM {TABLES['dim_content']}
""").df()

data = feb_agg.merge(mar_agg, on=['content_hash_id', 'client_hash_id'], how='inner')
data = data.merge(qmix, on='content_hash_id', how='left')
data = data.merge(age, on='content_hash_id', how='left')

data['content_age_days'] = (pd.Timestamp(MONTH + '-01') - pd.to_datetime(data['content_created_date'])).dt.days
data['is_down'] = (data['imp_mar'] < 0.8 * data['imp_feb']).astype(int)

print(f"Feature frame: {len(data):,} content items with enough Feb volume to score")
data[['imp_feb', 'pos_feb', 'content_age_days', 'visible_queries', 'rare_share', 'is_down']].head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 76,837 content items with enough Feb volume to score


,imp_feb,pos_feb,content_age_days,visible_queries,rare_share,is_down
0,1012.0,29.609070,227,25.0,0.054479,1
1,299.0,12.946228,227,2.0,0.140230,0
2,1598.0,17.806923,227,21.0,0.043740,0
3,514.0,10.490023,227,6.0,0.081583,1
4,2931.0,38.436254,227,27.0,0.027216,0


### e. The trap — feed the label's own ingredient back in

Honest features first, quick score noted. Then one deliberately leaky column: `imp_mar` — literally the number `is_down` is computed from.

In [27]:
from sklearn.tree import DecisionTreeClassifier, export_text

honest_features = ['imp_feb', 'pos_feb', 'content_age_days', 'visible_queries', 'rare_share']
model_data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=honest_features + ['imp_mar'])
X = model_data[honest_features]
y = model_data['is_down']

honest_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X, y)
honest_score = honest_tree.score(X, y)
print(f"Honest quick score (in-sample accuracy): {honest_score:.3f}")
print(export_text(honest_tree, feature_names=honest_features))

# Add the leak: imp_mar is the exact quantity is_down is computed from.
X_leaky = model_data[honest_features + ['imp_mar']]
leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X_leaky, y)
leaky_score = leaky_tree.score(X_leaky, y)
print(f"\nLeaky quick score (with imp_mar as a 'feature'): {leaky_score:.3f}  <- jumps toward perfect")
print(export_text(leaky_tree, feature_names=honest_features + ['imp_mar']))

# Delete the leak, keep the honest number.
print(f"\nRemoved imp_mar. Honest score stands: {honest_score:.3f}")


Honest quick score (in-sample accuracy): 0.651
|--- visible_queries <= 8.50
|   |--- content_age_days <= 72.50
|   |   |--- class: 0
|   |--- content_age_days >  72.50
|   |   |--- class: 1
|--- visible_queries >  8.50
|   |--- content_age_days <= 72.50
|   |   |--- class: 0
|   |--- content_age_days >  72.50
|   |   |--- class: 0


Leaky quick score (with imp_mar as a 'feature'): 0.797  <- jumps toward perfect
|--- imp_mar <= 351.50
|   |--- imp_feb <= 254.50
|   |   |--- class: 1
|   |--- imp_feb >  254.50
|   |   |--- class: 1
|--- imp_mar >  351.50
|   |--- imp_feb <= 508.50
|   |   |--- class: 0
|   |--- imp_feb >  508.50
|   |   |--- class: 0


Removed imp_mar. Honest score stands: 0.651


The leaky tree splits straight on `imp_mar` and nails the label — because `is_down` **is** a threshold on `imp_mar`. That's not signal, it's the answer smuggled in as a feature. `imp_mar` stays out of `honest_features`; the honest score above is the one that counts.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Window overlap in `fact_content_query_90d`.** That table is one fixed 90-day snapshot near the release's export date (2026-07-03), not a per-month rolling window. Used against a mid-panel label like March's, `visible_queries` and `rare_share` describe the content's query mix at release time — not certainly its March state. Treated as directional content traits here, not certified pre-March facts.

**Unbalanced panel.** History depth differs sharply per client — checked below against `dim_clients.gsc_data_start`. A March slice leans toward clients whose tracking started earlier, not a random cross-section of the full client base.

**GA4 is zero-filled, not missing-at-random**, before each client's `ga4_data_start` — confirmed in Section 3c. That's the reason it's excluded rather than imputed.

In [28]:
history_check = con.sql(f"""
    SELECT COUNT(*) AS n_clients,
           SUM(CASE WHEN gsc_data_start <= (SELECT MAX(gsc_data_start) FROM {TABLES['dim_clients']}) - INTERVAL 365 DAY
                THEN 1 ELSE 0 END) AS n_12mo_plus
    FROM {TABLES['dim_clients']}
""").df()
history_check


,n_clients,n_12mo_plus
0,104,4.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.